In [1]:
import folium
import wget
import pandas as pd

In [2]:
from folium.plugins import MarkerCluster

from folium.plugins import MousePosition

from folium.features import DivIcon

In [5]:
spacex_csv_file =r"C:\Users\parja\OneDrive\Desktop\DATASETS\spacex_launch_geo.csv"
spacex_df=pd.read_csv(spacex_csv_file)

In [6]:
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [7]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

In [9]:
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))

marker = folium.map.Marker(
    nasa_coordinate,
    
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

In [10]:

site_map = folium.Map(
    location=nasa_coordinate,
    zoom_start=5
)


for index, row in spacex_df.iterrows():

    
    coordinate = [row['Lat'], row['Long']]

   
    folium.Circle(
        location=coordinate,
        radius=1000,
        color='#d35400',
        fill=True
    ).add_to(site_map)

    
    folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' 
                 % row['Launch Site']
        )
    ).add_to(site_map)


site_map

In [11]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


In [12]:
marker_cluster = MarkerCluster()


In [13]:
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'
    
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


In [14]:
site_map.add_child(marker_cluster)


for index, record in spacex_df.iterrows():
   
    marker_cluster.add_child(marker)

site_map

In [15]:
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

In [16]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

In [17]:
launch_site = spacex_df.iloc[0]


launch_site_lat = launch_site['Lat']
launch_site_lon = launch_site['Long']

print(launch_site_lat, launch_site_lon)

28.56230197 -80.57735648


In [18]:
coastline_lat = 28.56367
coastline_lon = -80.57163


distance_coastline = calculate_distance(
    launch_site_lat,
    launch_site_lon,
    coastline_lat,
    coastline_lon
)

print(distance_coastline)

0.5797581813109574


In [19]:
coordinate = [coastline_lat, coastline_lon]


distance_marker = folium.Marker(
    coordinate,

    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),

        html='<div style="font-size: 12; color:#d35400;">'
             '<b>{:.2f} KM</b></div>'.format(distance_coastline)
    )
)


distance_marker.add_to(site_map)

In [20]:
line_coordinates = [
    [launch_site_lat, launch_site_lon],
    [coastline_lat, coastline_lon]
]


folium.PolyLine(
    locations=line_coordinates,
    weight=2,
    color='blue'
).add_to(site_map)


site_map

In [21]:
railway_lat = 28.57250
railway_lon = -80.58520


distance_railway = calculate_distance(
    launch_site_lat,
    launch_site_lon,
    railway_lat,
    railway_lon
)

print(distance_railway)

1.3688628252722668


In [22]:
coordinate = [railway_lat, railway_lon]


distance_marker = folium.Marker(
    coordinate,

    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),

        html='<div style="font-size: 12; color:blue;">'
             '<b>{:.2f} KM</b></div>'.format(distance_railway)
    )
)

distance_marker.add_to(site_map)

In [23]:
folium.PolyLine(
    locations=[
        [launch_site_lat, launch_site_lon],
        [railway_lat, railway_lon]
    ],
    weight=2,
    color='blue'
).add_to(site_map)

site_map